# 02 -- IVIM Noise Sensitivity Analysis

**Corresponds to:** Manuscript Sec.2.2 (Model Validation), Supplemental Methods

This notebook evaluates the impact of Rician noise on IVIM model fitting and derived diffusion metrics across a range of signal-to-noise ratios (SNR). By adding controlled levels of Rician noise (SNR 5--80 dB) to a known ground-truth signal, we characterize the bias and variance introduced by noise in the estimated volume fractions (fw, pf, tissue) and DTI metrics (FA, MD).

**Key design:**
- Ground-truth signal generated from a fixed three-compartment IVIM model (fw=0.10, pf=0.05, tissue=0.85)
- Rician noise added via direct Gaussian noise on real/imaginary channels
- IVIM model fitting with fixed lambda_iso parameters
- Separate estimation of DTI metrics from the fitted cylinder compartment
- Systematic sweep across multiple SNR levels

## Setup & Imports

In [ ]:
import os, csv
import time
import numpy as np
from scipy.stats import rice
import nibabel as nib

from dmipy.signal_models import cylinder_models, gaussian_models
from dmipy.core import modeling_framework
from dmipy.core.modeling_framework import MultiCompartmentModel
from dmipy.core.acquisition_scheme import acquisition_scheme_from_bvalues, gtab_dmipy2dipy
from dmipy.utils import utils

from dipy.io import read_bvals_bvecs
from dipy.io.image import load_nifti
from dipy.core.gradients import gradient_table
from dipy.segment.mask import applymask, bounding_box, crop
from dipy.reconst.dti import TensorModel

import warnings
warnings.filterwarnings('ignore')
np.random.seed(41)

## 1. Configuration & Data Loading

Load NIfTI data, mask, and acquisition scheme.

In [ ]:
# ============================================================
# Configurable Parameters
# ============================================================
SNR_MIN = 5
SNR_MAX = 80
NUM_SNR_VALUES = 5
SIGMA = 1.0
OUTPUT_FILENAME = "ivim_output.csv"

# --- File paths ---
input_directory = r"C:\Users\arjun\PycharmProjects\DiffusionImagingTesting\input_data\slice"
output_directory = r"C:\Users\arjun\PycharmProjects\DiffusionImagingTesting\output_data\full_brain_noise"

fivim = input_directory + "/slice2.nii.gz"
fmask = input_directory + "/mask.nii.gz"
fbval = input_directory + "/bvals.txt"
fbvec = input_directory + "/bvecs.txt"

# ============================================================
# Load Data and Acquisition Scheme
# ============================================================
bvals, bvecs = read_bvals_bvecs(fbval, fbvec)
gtab = gradient_table(bvals, bvecs)
bvalues_SI = bvals * 1e6  # Convert to s/m^2

delta = 0.0106
Delta = 0.0431
scheme_ivim = acquisition_scheme_from_bvalues(
    bvalues_SI, bvecs, delta, Delta, b0_threshold=1e6, min_b_shell_distance=1e7
)

# Load NIfTI and mask
img_data, affine = load_nifti(fivim)
mask_data = nib.load(fmask).get_fdata()
mask_boolean = mask_data > 0.01
mins, maxs = bounding_box(mask_boolean)
mask_boolean = crop(mask_boolean, mins, maxs)
cropped_volume = crop(img_data, mins, maxs)
data = applymask(cropped_volume, mask_boolean)

print(f"Loaded data shape: {img_data.shape}")
print(f"Masked voxels: {data.shape[0]}")
print(f"Acquisition scheme: {scheme_ivim.nmr_echo_parameters.shape[0]} measurements")

## 2. Ground Truth Signal Generation

Generate the noiseless multi-compartment signal using fixed ground-truth parameters.

In [ ]:
# Model Setup
ball1 = gaussian_models.G1Ball()
ball2 = gaussian_models.G1Ball()
cyl = cylinder_models.C2CylinderStejskalTannerApproximation()
ballcyl = modeling_framework.MultiCompartmentModel([ball1, ball2, cyl])

# Ground truth parameters
params = {
    'G1Ball_1_lambda_iso': 3e-9,
    'G1Ball_2_lambda_iso': 7e-9,
    'C2CylinderStejskalTannerApproximation_1_lambda_par': 1.7e-9,
    'C2CylinderStejskalTannerApproximation_1_diameter': 1e-6,
    'C2CylinderStejskalTannerApproximation_1_mu': [0.4, 0.4],
    'partial_volume_0': 0.10,
    'partial_volume_1': 0.05,
    'partial_volume_2': 0.85
}

dummy_data = ballcyl(scheme_ivim, **params)
dummy_data = np.tile(dummy_data, [1, 1])
print(f"Ground truth signal shape: {dummy_data.shape}")
print(f"Ground truth: fw={params['partial_volume_0']}, pf={params['partial_volume_1']}, tissue={params['partial_volume_2']}")

## 3. Rician Noise Addition

Two implementations are provided. The primary method (`add_rician_noise_direct`) adds Gaussian noise to real and imaginary channels before magnitude reconstruction, which is the most physically accurate model for MRI magnitude data.

In [ ]:
def add_rician_noise(data, snr_db, sigma=1.0, n_repeats=18, extra_copies=1519):
    """
    Adds Rician noise to an existing dataset with a specified SNR (in dB).
    """
    v = np.sqrt(2 * sigma ** 2 * 10 ** (snr_db / 10))
    base = np.tile(data, [10, 1])
    noisy_data = rice.rvs(v / sigma, scale=sigma, size=base.shape)

    for _ in range(n_repeats):
        y = rice.rvs(v / sigma, scale=sigma, size=base.shape)
        noisy_data = np.append(noisy_data, y, axis=0)

    for _ in range(extra_copies):
        noisy_data = np.append(noisy_data, base, axis=0)

    # Empirical SNR check
    signal_power = np.mean(v ** 2)
    noise_power = 2 * sigma ** 2
    empirical_snr = 10 * np.log10(signal_power / noise_power)
    print(f"Target SNR: {snr_db:.2f} dB | Empirical SNR: {empirical_snr:.2f} dB")
    return noisy_data


def add_rician_noise_direct(data, snr_db, n_repeats=18, extra_copies=1519):
    """
    Adds Rician noise using direct Gaussian noise on real and imaginary channels.
    This is the most physically accurate representation.
    """
    S0 = np.max(data)
    sigma = S0 / (10 ** (snr_db / 20))

    noisy_data_list = []

    for _ in range(n_repeats + 1):
        # Add Gaussian noise to real and imaginary channels
        real = data + np.random.normal(0, sigma, data.shape)
        imag = np.random.normal(0, sigma, data.shape)
        # Take magnitude (this creates Rician distribution)
        noisy = np.sqrt(real ** 2 + imag ** 2)
        noisy_data_list.append(noisy)

    # Add clean copies
    for _ in range(extra_copies):
        noisy_data_list.append(data)

    noisy_data = np.vstack(noisy_data_list)

    # Verify SNR
    first_noisy = noisy_data_list[0]
    signal_power = np.mean(data ** 2)
    noise_power = np.mean((first_noisy - data) ** 2)

    if noise_power > 0:
        empirical_snr = 10 * np.log10(signal_power / noise_power)
    else:
        empirical_snr = np.inf

    print(f"Target SNR: {snr_db:.2f} dB | Empirical SNR: {empirical_snr:.2f} dB")
    print(f"Calculated sigma: {sigma:.6f}")
    print(f"Output shape: {noisy_data.shape}")

    return noisy_data

## 4. Fitting Helpers

In [ ]:
def fit_ivim(scheme, data):
    ivim_mod = MultiCompartmentModel([ball1, ball2, cylinder_models.C2CylinderStejskalTannerApproximation()])
    ivim_mod.set_fixed_parameter('G1Ball_1_lambda_iso', 3e-9)
    ivim_mod.set_fixed_parameter('G1Ball_2_lambda_iso', 7e-9)
    ivim_mod.set_fixed_parameter('C2CylinderStejskalTannerApproximation_1_lambda_par', 1.7e-9)
    ivim_fit = ivim_mod.fit(acquisition_scheme=scheme, data=data)
    return ivim_fit.fitted_parameters


def fit_dti_metrics(scheme, signal):
    gtab_dipy = gtab_dmipy2dipy(scheme)
    tenmod = TensorModel(gtab_dipy)
    tenfit = tenmod.fit(signal)
    return {
        'fa_mean': np.mean(tenfit.fa),
        'md_mean': np.mean(tenfit.md),
        'fa_std': np.std(tenfit.fa),
        'md_std': np.std(tenfit.md)
    }


def print_metrics(label, metrics):
    print(f"\n--- {label} ---")
    for k, v in metrics.items():
        print(f"{k}: {v:.6f}")

## 5. Multi-SNR Simulation

We sweep across SNR values from 5 dB to 80 dB. At each level:
1. Add Rician noise to the ground-truth signal
2. Fit the IVIM model 
3. Extract the cylinder compartment and compute DTI metrics
4. Compare fitted metrics against ground-truth values
5. Save results to CSV

In [ ]:
# Prepare CSV Writer
os.makedirs(output_directory, exist_ok=True)
output_csv = os.path.join(output_directory, OUTPUT_FILENAME)
csv_fieldnames = [
    "SNR",
    "fw_mean", "fw_std",
    "pf_mean", "pf_std",
    "tissue_mean", "tissue_std",
    "fitted_fa_mean", "fitted_md_mean", "fitted_fa_std", "fitted_md_std",
    "gt_fa_mean", "gt_md_mean", "gt_fa_std", "gt_md_std"
]
csv_file = open(output_csv, "w", newline='')
writer = csv.DictWriter(csv_file, fieldnames=csv_fieldnames)
writer.writeheader()

# Run Simulation for Multiple SNR Values
snr_values = np.linspace(SNR_MIN, SNR_MAX, NUM_SNR_VALUES)

for snr in snr_values:
    st = time.time()
    print("\n" + "="*50)
    print(f"Running simulation for SNR = {snr:.2f} dB")
    print("="*50)

    noisy_data = add_rician_noise_direct(dummy_data, snr)
    fitted_params = fit_ivim(scheme_ivim, noisy_data)

    ivim_metrics = {
        'fw_mean': np.mean(fitted_params['partial_volume_0']),
        'fw_std': np.std(fitted_params['partial_volume_0']),
        'pf_mean': np.mean(fitted_params['partial_volume_1']),
        'pf_std': np.std(fitted_params['partial_volume_1']),
        'tissue_mean': np.mean(fitted_params['partial_volume_2']),
        'tissue_std': np.std(fitted_params['partial_volume_2'])
    }

    # Fitted cylinder DTI metrics
    cyl_only_model = modeling_framework.MultiCompartmentModel([cyl])
    cyl_params_fitted = {
        'C2CylinderStejskalTannerApproximation_1_mu': fitted_params['C2CylinderStejskalTannerApproximation_1_mu'],
        'C2CylinderStejskalTannerApproximation_1_diameter': fitted_params['C2CylinderStejskalTannerApproximation_1_diameter'],
        'C2CylinderStejskalTannerApproximation_1_lambda_par': 1.7e-9
    }
    signal_cyl_only = cyl_only_model.simulate_signal(scheme_ivim, cyl_params_fitted)
    dti_fitted_metrics = fit_dti_metrics(scheme_ivim, signal_cyl_only)

    # Ground-truth DTI metrics
    cyl_params_gt = {
        'C2CylinderStejskalTannerApproximation_1_mu': [0.4, 0.4],
        'C2CylinderStejskalTannerApproximation_1_diameter': 1e-6,
        'C2CylinderStejskalTannerApproximation_1_lambda_par': 1.7e-9
    }
    signal_gt = cyl_only_model.simulate_signal(scheme_ivim, cyl_params_gt)
    dti_groundtruth_metrics = fit_dti_metrics(scheme_ivim, signal_gt)

    # Combine all metrics and write row
    all_metrics = {
        "SNR": snr,
        **ivim_metrics,
        **{f"fitted_{k}": v for k, v in dti_fitted_metrics.items()},
        **{f"gt_{k}": v for k, v in dti_groundtruth_metrics.items()}
    }
    writer.writerow(all_metrics)

    print_metrics("IVIM Metrics", ivim_metrics)
    print_metrics("Fitted DTI", dti_fitted_metrics)
    print_metrics("Ground Truth DTI", dti_groundtruth_metrics)

    et = time.time()
    print(f"Completed SNR = {snr:.2f} dB in {et-st:.2f} seconds")

csv_file.close()
print(f"\nAll results saved to: {output_csv}")

## 6. Quick Summary

Display a summary of the fitted metrics across SNR values.

In [ ]:
import pandas as pd

summary = pd.read_csv(output_csv)
print("\nSummary of multi-SNR simulation:")
for _, row in summary.iterrows():
    print(f"  SNR={row['SNR']:5.1f} dB | fw={row['fw_mean']:.4f}+/-{row['fw_std']:.4f} | "
          f"pf={row['pf_mean']:.4f}+/-{row['pf_std']:.4f} | "
          f"FA={row['fitted_fa_mean']:.4f}+/-{row['fitted_fa_std']:.4f}")

print("\n✓ Noise sensitivity analysis complete.")